<a href="https://colab.research.google.com/github/djduncanlaw/AISV-819-LLM-Fundamentals-and-Practical-Applications/blob/main/OpenClaw_AI_News_Digest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### OpenClaw AI News Digest Setup

This notebook sets up the environment to use the OpenClaw agent on Google Cloud Platform to fetch AI news from Google News and email a digest via Gmail, based on the provided guide.

In [ ]:
# Install required libraries
!pip install feedparser google-cloud-storage google-cloud-pubsub -q

### Prerequisites & Configuration Setup

Before running the code, you need to gather some credentials. Please update the `config.json` file (which will be generated in the next cell) with the following information:

1. **GCP Project ID (`PROJECT_ID`)**:
   * Go to the [Google Cloud Console](https://console.cloud.google.com/).
   * Click the project drop-down at the top and select **New Project** (or use an existing one).
   * Copy the **Project ID** (not the Project Name).
   * *Note: You will also need to update the `PROJECT_ID` variable inside the bash scripts (`deploy_openclaw.sh`, `verify_openclaw_vm.sh`, `cleanup_openclaw.sh`) below to match this.*

2. **Gmail App Password (`APP_PASSWORD`)**:
   * Go to your [Google Account Security settings](https://myaccount.google.com/security).
   * Ensure **2-Step Verification** is turned ON.
   * Search for **App passwords** in the security settings search bar.
   * Create a new App password (you can name it something like "Colab OpenClaw").
   * Copy the 16-character password provided. **Never use your actual Google account password.**

3. **GCP Authentication**:
   * To execute the Google Cloud commands in this notebook, you must authenticate your session. You can do this by running `!gcloud auth login` in a separate cell if prompted.


In [ ]:
import os
import json

# 1. Create a template configuration file (edit config.json in the file explorer)
config_template = {
    "PROJECT_ID": "your-gcp-project-id",
    "SENDER_EMAIL": "your_email@gmail.com",
    "APP_PASSWORD": "your_gmail_app_password",
    "RECEIVER_EMAIL": "receiver_email@gmail.com"
}

# Write the template to a file (only runs if file doesn't exist to prevent overwriting your edits)
if not os.path.exists('config.json'):
    with open('config.json', 'w') as f:
        json.dump(config_template, f, indent=4)
    print("Created config.json template. Please update it with your real credentials in the file explorer.")

# 2. Load the configuration file
with open('config.json', 'r') as f:
    config = json.load(f)

PROJECT_ID = config.get('PROJECT_ID')
SENDER_EMAIL = config.get('SENDER_EMAIL')
APP_PASSWORD = config.get('APP_PASSWORD')
RECEIVER_EMAIL = config.get('RECEIVER_EMAIL')

print("Loaded configuration from config.json")

# 3. Authenticate with Google Cloud (Seamless for Colab / Colab Enterprise)
try:
    from google.colab import auth
    print("Authenticating with Google Cloud...")
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Authentication successful.")
except ImportError:
    print("Not running in Colab. Skipping native authentication.")

# Set GCP Project
!gcloud config set project {PROJECT_ID}

Loaded configuration from config.json
Authenticating with Google Cloud...
Authentication successful.
Updated property [core/project].


In [ ]:
import json
import os

if os.path.exists('config.json'):
    with open('config.json', 'r') as f:
        current_config = json.load(f)
        print("--- Current config.json contents ---")
        print(json.dumps(current_config, indent=4))
else:
    print("config.json does not exist yet. Please run the previous cell to create it.")

--- Current config.json contents ---
{
    "PROJECT_ID": "idyllic-theater-475721-f2",
    "SENDER_EMAIL": "capstone.fall25@gmail.com",
    "APP_PASSWORD": "nyuk cimo nebc alor",
    "RECEIVER_EMAIL": "capstone.fall25@gmail.com"
}


In [ ]:
# Authenticate your Google Cloud session
!gcloud auth login


You are running on a Google Compute Engine virtual machine.
It is recommended that you use service accounts for authentication.

You can run:

  $ gcloud config set account `ACCOUNT`

to switch accounts if necessary.

Your credentials may be visible to others with access to this
virtual machine. Are you sure you want to authenticate with
your personal account?

Do you want to continue (Y/n)?  Y

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.

### AI News Fetching and OpenClaw Agent execution

We define a function to fetch the news and then use OpenClaw (simulated here with standard Python to fit the Colab environment) to process and send it.

In [ ]:
import feedparser
from datetime import datetime
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from IPython.display import HTML, display
import re
import textwrap

# In a full OpenClaw setup, this would be an Agent Task.
def fetch_ai_news(num_articles=10):
    print("Fetching AI news from Google News...")
    rss_url = 'https://news.google.com/rss/search?q=Artificial+Intelligence+when:1d&hl=en-US&gl=US&ceid=US:en'
    feed = feedparser.parse(rss_url)

    digest = f"<h2>AI News Digest - {datetime.now().strftime('%Y-%m-%d')}</h2>\n"
    for i, entry in enumerate(feed.entries[:num_articles]):
        raw_summary = entry.get('summary', 'No summary available.')
        # Strip HTML tags to avoid redundant links
        clean_summary = re.sub(r'<[^>]+>', '', raw_summary)
        # Truncate to approximately 2 lines (200 characters)
        short_summary = textwrap.shorten(clean_summary, width=200, placeholder="...")
        digest += f"<p><strong>{i+1}. <a href='{entry.link}'>{entry.title}</a></strong><br><em>{short_summary}</em></p>\n"
    return digest

def send_email(digest_content):
    print("Sending email digest...")
    msg = MIMEMultipart()
    msg['From'] = SENDER_EMAIL
    msg['To'] = RECEIVER_EMAIL
    msg['Subject'] = f"Daily AI News Digest - {datetime.now().strftime('%Y-%m-%d')}"
    msg.attach(MIMEText(digest_content, 'html'))

    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(SENDER_EMAIL, APP_PASSWORD)
        server.send_message(msg)
        server.quit()
        print("Successfully sent the AI News Digest email!")
    except Exception as e:
        print(f"Failed to send email. Error: {e}")

# Execute the task
news_digest = fetch_ai_news(10)
display(HTML(news_digest))
send_email(news_digest)

Fetching AI news from Google News...


Sending email digest...
Successfully sent the AI News Digest email!


### Automate OpenClaw GCP Implementation

The following script automates the provisioning of the OpenClaw Gateway on Google Cloud Platform using the official deployment steps.

**Note:** You must be authenticated with Google Cloud (`gcloud auth login`) before executing this script.

In [ ]:
# --- Deploy OpenClaw GCP Resources ---
VM_NAME="openclaw-gateway"
ZONE="us-central1-a"
SERVICE_ACCOUNT="openclaw-deploy"

print("=========================================")
print("Starting OpenClaw GCP Deployment...")
print("=========================================")

print(f"\n[1/5] Setting up GCP Project ({PROJECT_ID})...")
!gcloud config set project {PROJECT_ID}

print("\n[2/5] Enabling Compute Engine API...")
!gcloud services enable compute.googleapis.com

print("\n[3/5] Creating OpenClaw Gateway VM instance...")
!gcloud compute instances create {VM_NAME} \
  --zone={ZONE} \
  --machine-type=e2-small \
  --boot-disk-size=20GB \
  --image-family=debian-12 \
  --image-project=debian-cloud

print(f"\n[4/5] Creating Service Account ({SERVICE_ACCOUNT})...")
!gcloud iam service-accounts create {SERVICE_ACCOUNT} \
  --display-name="OpenClaw Deployment"

print("\n[5/5] Assigning IAM roles to the Service Account...")
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
  --member="serviceAccount:{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com" \
  --role="roles/compute.instanceAdmin.v1"

print("\n=========================================")
print("OpenClaw GCP Deployment automation complete!")
print("=========================================")

Starting OpenClaw GCP Deployment...

[1/5] Setting up GCP Project (idyllic-theater-475721-f2)...
Updated property [core/project].

[2/5] Enabling Compute Engine API...

[3/5] Creating OpenClaw Gateway VM instance...
ERROR: (gcloud.compute.instances.create) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' already exists


[4/5] Creating Service Account (openclaw-deploy)...
ERROR: (gcloud.iam.service-accounts.create) Resource in projects [idyllic-theater-475721-f2] is the subject of a conflict: Service account openclaw-deploy already exists within project projects/idyllic-theater-475721-f2.
- '@type': type.googleapis.com/google.rpc.ResourceInfo
  resourceName: projects/idyllic-theater-475721-f2/serviceAccounts/openclaw-deploy@idyllic-theater-475721-f2.iam.gserviceaccount.com

[5/5] Assigning IAM roles to the Service Account...
Updated IAM policy for project [idyllic-theater-475721-f2].
bindings:
- members:
  - s

### Verify VM Deployment

Use this script to check if the OpenClaw Gateway VM is running and to find its external IP address.

In [ ]:
# --- Verify OpenClaw VM status ---
VM_NAME="openclaw-gateway"
ZONE="us-central1-a"

print("=========================================")
print(f"Checking status of VM: {VM_NAME}")
print("=========================================")

!gcloud config set project {PROJECT_ID} --quiet

!gcloud compute instances describe {VM_NAME} \
    --zone={ZONE} \
    --format="table(name, zone, status, machineType, networkInterfaces[0].accessConfigs[0].natIP)"

Checking status of VM: openclaw-gateway
Updated property [core/project].
NAME              ZONE                                                                                          STATUS   MACHINE_TYPE                                                                                                        NAT_IP
openclaw-gateway  https://www.googleapis.com/compute/v1/projects/idyllic-theater-475721-f2/zones/us-central1-a  RUNNING  https://www.googleapis.com/compute/v1/projects/idyllic-theater-475721-f2/zones/us-central1-a/machineTypes/e2-small  34.61.64.205


### Deploy OpenClaw Agent to VM (with Gemini Backend)

Now that the VM is running, we will deploy the actual OpenClaw agent software to it. This script will SSH into the VM, install the necessary dependencies, configure the agent to use **Gemini** as its LLM backend, and start the service in the background.

Make sure your `GEMINI_API_KEY` is saved in the Colab Secrets panel on the left.

In [ ]:
import os
import time
from google.colab import userdata

# Retrieve Gemini API Key from Colab Secrets
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("Successfully loaded GEMINI_API_KEY from secrets.")
except Exception as e:
    print("WARNING: GEMINI_API_KEY not found in Colab secrets. Please add it.")
    GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

# Create the deployment script that will run on the VM
deploy_script = f"""#!/bin/bash
echo "Updating package lists..."
sudo apt-get update -y

echo "Installing Python and pip..."
sudo apt-get install -y python3-pip git

echo "Installing OpenClaw agent dependencies..."
# Assuming openclaw is installed via pip for this environment
pip3 install openclaw google-generativeai --break-system-packages

echo "Configuring OpenClaw environment..."
export LLM_BACKEND="gemini"
export GEMINI_API_KEY="{GEMINI_API_KEY}"

echo "Starting OpenClaw agent..."
# Run the agent in the background and redirect output to a log file
nohup openclaw start --backend gemini > ~/openclaw_agent.log 2>&1 &

echo "OpenClaw agent deployed and started successfully!"
"""

# Write the script to a local file
with open("deploy_agent.sh", "w") as f:
    f.write(deploy_script)

print("\nWaiting 30 seconds to allow SSH keys to propagate to the VM...")
time.sleep(30)

print("\nTransferring deployment script to the VM...")
!gcloud compute scp deploy_agent.sh deployer@{VM_NAME}:~/deploy_agent.sh --zone={ZONE} --quiet

print("\nExecuting deployment script on the VM...")
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="chmod +x ~/deploy_agent.sh && ~/deploy_agent.sh" --quiet

print("\nDeployment command completed. You can verify the logs on the VM by running:")
print(f"!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command='cat ~/openclaw_agent.log'")


Successfully loaded GEMINI_API_KEY from secrets.

Waiting 30 seconds to allow SSH keys to propagate to the VM...

Transferring deployment script to the VM...
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)

Executing deployment script on the VM...
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
Updating package lists...
Get:1 file:/etc/apt/mirrors/debian.list Mirrorlist [30 B]
Get:5 file:/etc/apt/mirrors/debian-security.list Mirrorlist [39 B]
Hit:7 https://packages.cloud.google.com/apt google-compute-engine-bookworm-stable InRelease
Hit:8 https://packages.cloud.google.com/apt cloud-sdk-bookworm InRelease
Hit:9 https://deb.nodesource.com/node_22.x nodistro InRelease
Hit:10 https://packages.cloud.google.com/apt google-cloud-ops-agent-bookworm-2 InRelease
Hit:2 https://deb.debian.org/debian bookworm InRelease
Hit:3 https://deb.debian.org/debian bookworm-updates InRelease
Hit

### Run OpenClaw Agent Locally in Colab

Alternatively, you can run the OpenClaw agent directly in this Colab notebook using Gemini as the backend to achieve the objective, rather than deploying it to a GCP VM.

In [ ]:
# Install OpenClaw and Gemini dependencies locally in Colab
!pip install openclaw google-generativeai -q

In [ ]:
import os
from google.colab import userdata
import json

# Configure Environment for OpenClaw
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    os.environ['LLM_BACKEND'] = 'gemini'
    print("Environment configured for OpenClaw with Gemini backend.")
except Exception as e:
    print("Error loading GEMINI_API_KEY from secrets. Please ensure it is set.")

# Load credentials to pass to the agent
with open('config.json', 'r') as f:
    config = json.load(f)

agent_prompt = f"""
Your objective is to act as an automated news assistant.
1. Fetch the top 10 Artificial Intelligence news articles from Google News RSS.
2. Create a concise, two-line summary for each article.
3. Send an HTML formatted email digest with these summaries to {config['RECEIVER_EMAIL']} from {config['SENDER_EMAIL']} using SMTP.
Use the following Gmail app password for SMTP authentication: {config['APP_PASSWORD']}
"""

# Save the prompt to a file for the agent to consume
with open('agent_task.txt', 'w') as f:
    f.write(agent_prompt)

print("Agent task defined. Ready to execute.")

Environment configured for OpenClaw with Gemini backend.
Agent task defined. Ready to execute.


### Run Official Node.js OpenClaw on GCP VM
Since the Python package is broken, we will execute the official Node.js version of the OpenClaw agent directly on your `openclaw-gateway` VM.

In [ ]:
import json
import os
from google.colab import userdata

# Load credentials
with open('config.json', 'r') as f:
    config = json.load(f)

try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except:
    GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

VM_NAME = "openclaw-gateway"
ZONE = "us-central1-a"

# Bash script to run on the VM
node_script = f"""#!/bin/bash
echo "Upgrading to Node.js 22.x..."
curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - > /dev/null 2>&1
sudo apt-get install -y nodejs > /dev/null 2>&1

echo "Creating task prompt..."
cat << 'EOF' > ~/agent_task.txt
Your objective is to act as an automated news assistant.
1. Fetch the top 10 Artificial Intelligence news articles from Google News RSS.
2. Create a concise, two-line summary for each article.
3. Send an HTML formatted email digest with these summaries to {config['RECEIVER_EMAIL']} from {config['SENDER_EMAIL']} using SMTP.
Use the following Gmail app password for SMTP authentication: {config['APP_PASSWORD']}
EOF

echo "Installing OpenClaw globally via npm..."
sudo npm install -g openclaw@latest > /dev/null 2>&1

echo "Running OpenClaw agent..."
export GEMINI_API_KEY="{GEMINI_API_KEY}"
export LLM_BACKEND="gemini"

# Run the agent using the non-interactive single command mode
openclaw crestodian --message "$(cat ~/agent_task.txt)"
"""

# Save locally
with open('run_node_openclaw.sh', 'w') as f:
    f.write(node_script)

print("Transferring Node.js execution script to the VM...")
!gcloud compute scp run_node_openclaw.sh deployer@{VM_NAME}:~/run_node_openclaw.sh --zone={ZONE} --quiet

print("\nExecuting OpenClaw on the VM (this may take a moment to install and run)...")
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="chmod +x ~/run_node_openclaw.sh && ~/run_node_openclaw.sh" --quiet


Transferring Node.js execution script to the VM...
ERROR: (gcloud.compute.scp) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found


Executing OpenClaw on the VM (this may take a moment to install and run)...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
VM_NAME = "openclaw-gateway"
ZONE = "us-central1-a"

print("Asking OpenClaw to check and restart the Gateway...")
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command='export GEMINI_API_KEY="{GEMINI_API_KEY}"; export LLM_BACKEND="gemini"; openclaw crestodian --message "check and restart gateway" --yes' --quiet


Asking OpenClaw to check and restart the Gateway...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
# Automate the OpenClaw setup and Gateway installation on the VM
setup_script = """#!/bin/bash
echo "Running openclaw setup..."
# Use 'yes' to bypass any interactive prompts during setup
yes | openclaw setup

echo "Installing OpenClaw Gateway service..."
openclaw gateway install

echo "Starting OpenClaw Gateway service..."
systemctl --user daemon-reload
systemctl --user enable openclaw-gateway.service
systemctl --user start openclaw-gateway.service
systemctl --user status openclaw-gateway.service --no-pager
"""

with open('setup_gateway.sh', 'w') as f:
    f.write(setup_script)

print("Transferring setup script to the VM...")
!gcloud compute scp setup_gateway.sh deployer@{VM_NAME}:~/setup_gateway.sh --zone={ZONE} --quiet

print("\nExecuting setup script on the VM...")
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="chmod +x ~/setup_gateway.sh && ~/setup_gateway.sh" --quiet


Transferring setup script to the VM...
ERROR: (gcloud.compute.scp) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found


Executing setup script on the VM...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



### Cleanup OpenClaw GCP Resources

Run the script below if you need to tear down the environment or if the deployment fails. This helps prevent unexpected charges.

In [ ]:
%%script false --no-raise-error

# --- Cleanup OpenClaw GCP Resources ---
VM_NAME="openclaw-gateway"
ZONE="us-central1-a"
SERVICE_ACCOUNT="openclaw-deploy"

print("=========================================")
print("Starting cleanup of OpenClaw resources...")
print("=========================================")

print(f"\n[1/3] Deleting VM instance ({VM_NAME})...")
!gcloud compute instances delete {VM_NAME} --zone={ZONE} --quiet

print("\n[2/3] Removing IAM policy binding...")
!gcloud projects remove-iam-policy-binding {PROJECT_ID} \
  --member="serviceAccount:{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com" \
  --role="roles/compute.instanceAdmin.v1" --quiet

print(f"\n[3/3] Deleting Service Account ({SERVICE_ACCOUNT})...")
!gcloud iam service-accounts delete "{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com" --quiet

print("\n=========================================")
print("Cleanup complete!")
print("=========================================")

Starting cleanup of OpenClaw resources...

[1/3] Deleting VM instance (openclaw-gateway)...
Deleted [https://www.googleapis.com/compute/v1/projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway].

[2/3] Removing IAM policy binding...
Updated IAM policy for project [idyllic-theater-475721-f2].
bindings:
- members:
  - serviceAccount:service-557844001021@gcp-sa-vertex-nb.iam.gserviceaccount.com
  role: roles/aiplatform.colabServiceAgent
- members:
  - serviceAccount:service-557844001021@gcp-sa-aiplatform-vm.iam.gserviceaccount.com
  role: roles/aiplatform.notebookServiceAgent
- members:
  - serviceAccount:service-557844001021@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/aiplatform.serviceAgent
- members:
  - user:capstone.fall25@gmail.com
  role: roles/bigquery.studioAdmin
- members:
  - serviceAccount:557844001021@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-557844001021@gcp-sa-cloudbui

In [ ]:
print("=========================================")
print("Configuring Model and Executing Task on VM...")
print("=========================================")

# Configure model via agent message and run the news task
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command='export GEMINI_API_KEY="{GEMINI_API_KEY}"; export LLM_BACKEND="gemini"; openclaw crestodian --message "set default model to gemini" --yes && openclaw crestodian --message "$(cat ~/agent_task.txt)" --yes' --quiet


Configuring Model and Executing Task on VM...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("Fetching the latest OpenClaw session transcript from the VM...")

# Find the most recent session directory and print its contents (transcript or messages)
# Avoid curly braces in bash variables to prevent IPython string interpolation conflicts
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command='LATEST_SESSION=$(ls -td ~/.openclaw/agents/main/sessions/*/ 2>/dev/null | head -n 1); if [ -n "$LATEST_SESSION" ]; then echo "Latest Session: $LATEST_SESSION"; cat "$LATEST_SESSION"transcript.txt 2>/dev/null || cat "$LATEST_SESSION"messages.json 2>/dev/null || ls -la "$LATEST_SESSION"; else echo "No sessions found."; fi' --quiet


Fetching the latest OpenClaw session transcript from the VM...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("Listing contents of the agent's main directory on the VM...")
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="ls -laR ~/.openclaw/agents/main/" --quiet


Listing contents of the agent's main directory on the VM...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("Fixing the model configuration in openclaw.json on the VM...")
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="jq 'del(.defaultModel) | .agents.defaults.model.primary = \"gemini\" | .agents.defaults.models = {{\"gemini\": {{}}}}' ~/.openclaw/openclaw.json > ~/.openclaw/tmp.json && mv ~/.openclaw/tmp.json ~/.openclaw/openclaw.json && echo 'Configuration fixed. Current openclaw.json:' && cat ~/.openclaw/openclaw.json" --quiet


Fixing the model configuration in openclaw.json on the VM...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("\n=========================================")
print("Re-running the news digest task with the fixed configuration...")
print("=========================================")

!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command='export GEMINI_API_KEY="{GEMINI_API_KEY}"; export LLM_BACKEND="gemini"; openclaw crestodian --message "$(cat ~/agent_task.txt)" --yes' --quiet



Re-running the news digest task with the fixed configuration...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("=========================================")
print("Running OpenClaw doctor on the VM...")
print("=========================================")

!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command='openclaw doctor' --quiet


Running OpenClaw doctor on the VM...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("=========================================")
print("Fixing OpenClaw Config and Running Task...")
print("=========================================")

fix_and_run_script = f"""#!/bin/bash
# Clean up bad keys
jq 'del(.defaultModel)' ~/.openclaw/openclaw.json > ~/.openclaw/tmp.json && mv ~/.openclaw/tmp.json ~/.openclaw/openclaw.json

# Set the default model properly using the config command
openclaw config set agents.defaults.model.primary '"gemini"'

# Run the task
export GEMINI_API_KEY="{GEMINI_API_KEY}"
export LLM_BACKEND="gemini"
openclaw crestodian --message "$(cat ~/agent_task.txt)" --yes
"""

with open("fix_and_run.sh", "w") as f:
    f.write(fix_and_run_script)

!gcloud compute scp fix_and_run.sh deployer@{VM_NAME}:~/fix_and_run.sh --zone={ZONE} --quiet
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="chmod +x ~/fix_and_run.sh && ~/fix_and_run.sh" --quiet


Fixing OpenClaw Config and Running Task...
ERROR: (gcloud.compute.scp) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found

ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("=========================================")
print("Executing News Digest Task...")
print("=========================================")

task_script = f"""#!/bin/bash
# Ensure correct provider format to avoid warnings
openclaw config set agents.defaults.model.primary '"google/gemini"'

export GEMINI_API_KEY="{GEMINI_API_KEY}"
export LLM_BACKEND="gemini"

echo "Running OpenClaw agent..."
openclaw crestodian --message "$(cat ~/agent_task.txt)" --yes
"""

with open("run_task.sh", "w") as f:
    f.write(task_script)

!gcloud compute scp run_task.sh deployer@{VM_NAME}:~/run_task.sh --zone={ZONE} --quiet
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="chmod +x ~/run_task.sh && ~/run_task.sh" --quiet


Executing News Digest Task...
ERROR: (gcloud.compute.scp) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found

ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("=========================================")
print("Fetching Execution Logs...")
print("=========================================")

fetch_logs_script = """#!/bin/bash
echo "Checking latest OpenClaw session..."
LATEST_SESSION=$(ls -td ~/.openclaw/agents/main/sessions/*/ 2>/dev/null | head -n 1)

if [ -n "$LATEST_SESSION" ]; then
    echo "Found session: $LATEST_SESSION"
    echo "\n--- Transcript ---"
    cat "${LATEST_SESSION}transcript.txt" 2>/dev/null || echo "No transcript.txt found."
    echo "\n--- Messages ---"
    cat "${LATEST_SESSION}messages.json" 2>/dev/null || echo "No messages.json found."
else
    echo "No sessions found."
fi
"""

with open("fetch_logs.sh", "w") as f:
    f.write(fetch_logs_script)

!gcloud compute scp fetch_logs.sh deployer@{VM_NAME}:~/fetch_logs.sh --zone={ZONE} --quiet
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="chmod +x ~/fetch_logs.sh && ~/fetch_logs.sh" --quiet


Fetching Execution Logs...
ERROR: (gcloud.compute.scp) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found

ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("=========================================")
print("Executing News Digest Task (Default Agent)...")
print("=========================================")

task_script = f"""#!/bin/bash
export GEMINI_API_KEY="{GEMINI_API_KEY}"
export LLM_BACKEND="gemini"

echo "Restarting OpenClaw Gateway..."
systemctl --user restart openclaw-gateway.service
sleep 2

echo "Running OpenClaw default agent..."
openclaw "talk to agent" --message "$(cat ~/agent_task.txt)" --yes
"""

with open("run_main_task.sh", "w") as f:
    f.write(task_script)

!gcloud compute scp run_main_task.sh deployer@{VM_NAME}:~/run_main_task.sh --zone={ZONE} --quiet
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="chmod +x ~/run_main_task.sh && ~/run_main_task.sh" --quiet


Executing News Digest Task (Default Agent)...
ERROR: (gcloud.compute.scp) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found

ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found



In [ ]:
print("=========================================")
print("Restarting and Verifying OpenClaw Gateway...")
print("=========================================")

# Restart the gateway service and display its status
!gcloud compute ssh deployer@{VM_NAME} --zone={ZONE} --command="systemctl --user daemon-reload && systemctl --user restart openclaw-gateway.service && sleep 2 && systemctl --user status openclaw-gateway.service --no-pager" --quiet


Restarting and Verifying OpenClaw Gateway...
ERROR: (gcloud.compute.ssh) Could not fetch resource:
 - The resource 'projects/idyllic-theater-475721-f2/zones/us-central1-a/instances/openclaw-gateway' was not found

